# Partial Differential Equations Part III. 

So far we've discussed the
 [3 broad classes of PDEs](https://en.wikiversity.org/wiki/Partial_differential_equations#Elliptic,_Hyperbolic,_and_Parabolic_PDEs): hyperbolic, elliptic, and parabolic PDEs.  

And discussed the following methods for solving BVP PDEs:
+ Jacobi Method (Relaxation)
+ Gauss-Seidel 
+ Overrelaxation 

And the following for solving IVP PDEs:
+ FTCS
+ Upwinding

We found that FTCS works okay for parabolic PDEs, under certain conditions, but not so well so for hyperbolic PDEs for which we have to modify in terms of upwinding.

[Last week](./9a_FTs.ipynb), we went over how Fourier transforms can help us with understanding the stability of our schemes. 

## Ex: The Wave Equation
This week, we'll focus on comparing across our methods for the canonical hyperbolic PDE: the *wave equation*:

$$\frac{\partial^2 \phi}{\partial t^2} = c^2 \frac{\partial^2 \phi}{\partial x^2},$$

where we have derivatives in both space and time and typically the initial conditions for $\phi(x,0)$ on all $x$. 

Let's consider the wave equation for a wave on a fixed string of length $L$ and fixed at both ends such that the displacement: 
$$\phi(0) = \phi(L) = 0$$

The solution to this equation is a set of traveling waves moving at speed $c$ in both directions:
$$\phi(x, t) = A f_0(x - ct) + B g_0(x + ct)$$

### FTCS Stability Analysis
Let's use a Fourier decomposition to perform a *von Neumann stability analysis* to look at the stability of FTCS for the wave equation.

The FTCS discretization for the wave equation, where $\psi(x,t) = \frac{\partial \phi}{\partial t}$:
$$\phi(x, t+ \Delta t) = \phi(x,t) + \Delta x \psi(x,t)$$
$$\psi(x, t + \Delta t) = \psi(x,t) + \frac{\Delta t c^2}{\Delta x^2} [\phi(x+\Delta x,t)+\phi(x-\Delta x,t)-2\phi(x,t)]$$

From which we can write how the fourier coefficients evolve, for Fourier decompositions of  $\phi$ and $\psi$:
$$ c_{\phi}(t + \Delta t) = c_\phi (t) + \Delta t c_\psi(t)$$
$$ c_\psi (t + \Delta t) = c_\psi (t) - \Delta t \frac{4c^2}{\Delta x^{2}} \sin^2{\frac{k\Delta x}{2}}$$

Which we can express for the vector $\mathbf{c} = (c_\phi, c_\psi)$ as the following matrix operation: $\mathbf{c}(t + \Delta t) = \mathbf{A} \mathbf{c}(t)$

$$\mathbf{A} =  \left ( \begin{array}{cc}
                     1  &  \Delta t \\
                    -\Delta t \frac{4c^2}{\Delta x^{2}}\sin^2{\frac{k\Delta x}{2}}   &  1  \end{array} \right )
$$

Then the evolution of the coefficients can be expressed as the evolution of a linear combination of eigenvectors $v_1$ and $v_2$, with corresponding eigenvalues $\lambda_1$ and $\lambda_2$:

$$\mathbf{c}(t + \Delta t) = \alpha_1 \lambda_1 v_1 + \alpha_2 \lambda_2 v_2$$

Solving for the eigenvalues, we get:
$$|\mathbf{\lambda}| = \sqrt{1 + \frac{4c^2\Delta t^2}{\Delta x^{2}}\sin^2{\frac{\Delta x^2}{2}}}$$

Which is always $> 1$, that is, *regardless of the choice of timestep size, the numerical errors in the FTCS method for the wave equation will grow exponentially!*

The FTCS scheme for the wave equation is always numerically unstable, so we can only use it for very short integrations.

### Spectral Methods
We can express a solution for a single wavenumber $k$ in this alternative form:
$$\phi_k(x,t) = \sin\frac{\pi k x}{L}e^{i\omega t}$$

which automatically satisfies the boundary conditions and wave equation if:
$$\omega = \frac{\pi c k}{L}$$

Let's divide up the string to into $N$ equal intervals bounded by $N+1$ grid points, such that:
$$ x_n = \frac{n}{N} L$$

Then a general solution on points $x_n$ can be expressed as a linear combination of waves:

$$\phi(x_n, t) = \frac{1}{N} \sum_{k=1}^{N-1} b_k \sin\left ( \frac{\pi k n}{N}\right ) \exp \left (i \frac{\pi c k t}{L} \right )$$

with the complex-valued coefficients: $b_k = \alpha_k + i \eta_k$. In order to obtain the solution at any time, we then just need to find out the values of the coefficients.

To do so, we can set up to use our initial conditions. 

At $t=0$, the real-valued part of the solution becomes:
$$\phi(x_n, 0) = \frac{1}{N} \sum_{k=1}^{N-1} \alpha_k \sin\left ( \frac{\pi k n}{N}\right )$$

Similarly, applying the initial conditions for the real-valued part of the time derivative:
$$\dot{\phi}(x_n, 0) = -\frac{\pi c}{L N} \sum_{k=1}^{N-1} k \eta_k \sin\left ( \frac{\pi k n}{N}\right )$$

Thus, given, the value of the initial conditions for $\phi$ and $\dot{\phi}$, we can apply a *fast discrete sine transform* to compute the values of the coefficient $\alpha_k$ and $\eta_k$ and plug into the general solution and compute the sum directly.

In practice, the real part of the general solution can be expressed as another sine series:

$$\phi(x_n, t) = \frac{1}{N} \sum_{k=1}^{N-1} \left [ \alpha_k \cos\left ( \frac{\pi c k t}{L}\right ) - \eta_k \sin \left (\frac{\pi c k t}{L} \right ) \right ] \sin\left ( \frac{\pi k n}{N}\right )$$

This way, the spectral method doesn't require time-stepping, we solve the function for $\phi(x,t)$ at all times at once. To get the value at some $t$, we just need to evaluate this expression at any time $t$ we need!

The spectral method is always numerically stable.

```{note} Spectral Method Caveats
+ Only works for linear differential equations 
+ Requires simple BCs for which FTs are possible
+ Requires $N(\log N)$ calculations, while FTCS is $\mathcal{O}(N)$ for a single timestep
```

In practice, spectral methods can be generalized to a larger set of PDEs by a hybrid approach: applying spectral methods only in the spatial dimension and then using an ODE solver to integrate forward in time.   
This is the concept behind spectral method codes like [Dedalus](https://dedalus-project.org/)


### Implicit Methods
We can try to formulate an implicit version of FTCS, by writing the time reversed version (taking a step of $-\Delta t$):

$$\phi(x, t - \Delta t) = \phi(x,t) - \Delta x \psi(x,t)$$
$$\psi(x, t - \Delta t) = \psi(x,t) - \frac{\Delta t c^2}{\Delta x^2} [\phi(x+\Delta x,t)+\phi(x-\Delta x,t)-2\phi(x,t)]$$

Rearranging where $ t \rightarrow t + \Delta t$ and  $t - \Delta t \rightarrow t $, we can get the implicit in time, centered in space version:

$$\phi(x, t + \Delta t) - \Delta x \psi(x,t + \Delta t) = \phi(x,t) $$

$$\psi(x, t + \Delta t) - \frac{\Delta t c^2}{\Delta x^2} [\phi(x+\Delta x,t + \Delta t)+\phi(x-\Delta x,t+\Delta t)-2\phi(x,t+\Delta t)] = \psi(x,t) $$

A Von Neumann stability analysis for the coefficients $\mathbf{c} = (c_\phi,c_\psi)$ gives us the vector expression:

$$ {\mathbf B}\mathbf{c}(t + \Delta t) = \mathbf{c}(t)$$

which has a solution given by finding the inverse of the coefficient matrix:

$$ \mathbf{c}(t + \Delta t) = {\mathbf B}^{-1} \mathbf{c}(t)$$

$$\mathbf{B}^{-1} = \frac{1}{\Delta t^2 r^2 } \left ( \begin{array}{cc}
                     1  &  \Delta t \\
                    -\Delta t r^2  &  1  \end{array} \right )
$$

for $r^2 = (2c/\Delta x) \sin{(\frac{k \Delta x}{2})}$

You can find the eigenvalues as we did before to find:
$$|\mathbf{\lambda}| = \frac{1}{\sqrt{1 + \Delta t^2 r^2}}$$

which is *unconditionally stable* for the wave equation. 

However, the eigenvalues are independent of $k$ and will be $<1$ for all $k$, so each mode will decay over time: *a stable, but unphysical* solution for the wave equation.


### Crank-Nicolson 

So far, the Von Neumman stability analysis yields that, for the wave equation, we have:
+ FTCS: Fourier coefficients grow exponentially
+ Implicit: Fourier coefficients decay exponentially

So, what if we take the average of the two? $\rightarrow$ Crank-Nicolson!

$$\phi(x, t + \Delta t) - \Delta x \psi(x,t + \Delta t) = \phi(x,t) + \Delta x \psi(x,t)$$

$$\psi(x, t + \Delta t) - \frac{\Delta t c^2}{\Delta x^2} [\phi(x+\Delta x,t + \Delta t)+\phi(x-\Delta x,t+\Delta t)-2\phi(x,t+\Delta t)] = \psi(x,t)  - \frac{\Delta t c^2}{\Delta x^2} [\phi(x+\Delta x,t)+\phi(x-\Delta x,t)-2\phi(x,t)] $$


$$ {\mathbf B}\mathbf{c}(t + \Delta t) = \mathbf{A}\mathbf{c}(t)$$

which has a solution given by finding the inverse of the coefficient matrix:

$$ \mathbf{c}(t + \Delta t) = {\mathbf B}^{-1}\mathbf{A} \mathbf{c}(t)$$

:question:
```{embed} ex_10-1
```

```{embed} sol_10-1
```

The eigenvalues yield:

$$|\mathbf{\lambda}| = \frac{\sqrt{(1 - \Delta t^2 r^2 + 2i\Delta t r)(1 - \Delta t^2 r^2 - 2i\Delta t r)}}{1 + \Delta t^2 r^2} = 1$$

Exactly one! Waves don't grow or decay, but propagate!


## Ex: The Time-Dependent Schrodinger Equation 
Let's look at the Schrodinger equation in one dimension for a particle of mass $m$ with no potential energy:  

$$
-\frac{\hbar^2}{2m} \frac{\partial^2\psi}{\partial x^2}
  = i\hbar \frac{\partial\psi}{\partial t}.
$$

For a particle in a box with impenetrable walls:

$\psi(0) = \psi(L) = 0$ at all times $t$.

Discretizing our solution in space, we can write a vector $\mathbf{\psi} = [\psi_0, \ldots, \psi_N]$ to contain the value of $\psi_i$ at each point in space. 

The Crank-Nicolson Equations can then be written in the form:

$$\mathbf{A}\mathbf{\psi}(t + \Delta t) = \mathbf{B} \mathbf{\psi}(t)$$

where $\mathbf{A}$ and $\mathbf{B}$ are both symmetric and tri-diagonal:

$$
\mathbf{A} = \left [ \begin{array}{cccccc} a_1 & a_2 \\
                          a_2 & a_1 & a_2 \\
                              & a_2 & a_1 & a_2 \\
                              &     & a_2 & a_1 \\
                              &     &     &     & \ddots
          \end{array} \right ],\qquad\qquad
\mathbf{B} = \left [ \begin{array}{cccccc} b_1 & b_2 \\
                          b_2 & b_1 & b_2 \\
                              & b_2 & b_1 & b_2 \\
                              &     & b_2 & b_1 \\
                              &     &     &     & \ddots
          \end{array}\right ],
$$
with
$$
a_1 = 1 + \Delta t \frac{i\hbar}{2m\Delta x^2},\qquad
a_2 = - \Delta t \frac{i\hbar}{4m\Delta x^2},\qquad
b_1 = 1 - \Delta t \frac{i\hbar}{2m\Delta x^2},\qquad
b_2 = \Delta t \frac{i\hbar}{4m\Delta x^2}.
$$

As before, $x_n = \frac{n \Delta x}{N}$, describes our position in space in terms of our grid index.

The equation $\mathbf{A}\mathbf{\psi}(t+h) = \mathbf{B}\mathbf{\psi}(t)$ has precisely the form $\mathbf{A}\mathbf{x} = \mathbf{v}$ of the simultaneous equation problems from week 5 and can be solved using the same methods.  

Specifically, since the matrix $\mathbf{A}$ is tridiagonal in this case, we can use the fast tridiagonal version of Gaussian elimination in Newman Section 6.1.6.


<h2><span class="fa fa-flash"></span> In-Class Coding Exercise </h2>

```{embed} icc_10-1
```